# 📋 Structured Output with Pydantic

**Get reliable, validated JSON from LLMs**

---

## 📋 Overview

**What you'll learn:**
- Force JSON output from LLMs
- Validate with Pydantic models
- Handle parsing errors
- Create type-safe outputs
- Production patterns

**Time estimate:** ⏱️ 45 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, Field, validator
from typing import List, Optional
import json

load_dotenv()
client = Groq(api_key=os.getenv('GROQ_API_KEY'))

print("✅ Setup complete")

## 🎯 Why Structured Output?

**Problem:**
```python
response = "The product costs about $50 maybe?"
# Hard to parse! Is it $50? Uncertain?
```

**Solution:**
```python
response = {"price": 50.0, "confidence": "medium"}
# Easy to parse! Type-safe!
```

**Benefits:**
- ✅ Predictable format
- ✅ Automatic validation
- ✅ Type safety
- ✅ Easy integration with code

## 📊 Pydantic Models for LLM Output

In [ ]:
class ProductInfo(BaseModel):
    """Structured product information."""
    name: str = Field(..., description="Product name")
    price: float = Field(..., ge=0, description="Price in USD")
    category: str = Field(..., description="Product category")
    in_stock: bool = Field(..., description="Availability")
    features: List[str] = Field(default_factory=list, description="Key features")
    
    @validator('price')
    def price_reasonable(cls, v):
        if v > 1000000:
            raise ValueError('Price too high')
        return v

# Test validation
try:
    product = ProductInfo(
        name="Laptop",
        price=999.99,
        category="Electronics",
        in_stock=True,
        features=["16GB RAM", "SSD"]
    )
    print("✅ Valid product:")
    print(json.dumps(product.dict(), indent=2))
except Exception as e:
    print(f"❌ Validation error: {e}")

## 🔧 Extract Structured Data from LLM

In [ ]:
def extract_structured(text: str, model_class: type[BaseModel]) -> BaseModel:
    """Extract structured data using LLM."""
    # Get schema for the model
    schema = model_class.schema()
    
    prompt = f"""Extract information from the text and return ONLY valid JSON.

Required JSON schema:
{json.dumps(schema, indent=2)}

Text: {text}

JSON (no explanation, just JSON):"""
    
    response = client.chat.completions.create(
        model="mixtral-8x7b-32768",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=300
    )
    
    # Parse and validate
    json_str = response.choices[0].message.content
    data = json.loads(json_str)
    return model_class(**data)

# Test extraction
text = """The SuperLaptop Pro is available for $1,299. 
It's an electronics item currently in stock with 16GB RAM and 512GB SSD."""

try:
    product = extract_structured(text, ProductInfo)
    print("✅ Extracted product:")
    print(json.dumps(product.dict(), indent=2))
except Exception as e:
    print(f"❌ Error: {e}")

## 🏗️ Production-Ready Extractor

In [ ]:
class StructuredExtractor:
    """Production extractor with retry and validation."""
    
    def __init__(self):
        self.client = Groq(api_key=os.getenv('GROQ_API_KEY'))
    
    def extract(self, text: str, model_class: type[BaseModel], 
                max_retries: int = 3) -> Optional[BaseModel]:
        """Extract with retries."""
        for attempt in range(max_retries):
            try:
                schema = model_class.schema()
                
                prompt = f"""Extract to JSON (schema below):
{json.dumps(schema, indent=2)}

Text: {text}

JSON:"""
                
                response = self.client.chat.completions.create(
                    model="mixtral-8x7b-32768",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0,
                    max_tokens=500
                )
                
                json_str = response.choices[0].message.content
                data = json.loads(json_str)
                return model_class(**data)
                
            except Exception as e:
                if attempt == max_retries - 1:
                    raise
                print(f"Attempt {attempt + 1} failed, retrying...")
        
        return None

# Test
extractor = StructuredExtractor()
result = extractor.extract(
    "The iPhone 15 costs $999 and is in the phones category. Currently in stock.",
    ProductInfo
)

if result:
    print("Extracted:")
    print(json.dumps(result.dict(), indent=2))

## ✅ Summary

### Key Points:
- 📋 **Pydantic**: Validate LLM output
- 🔒 **Type-safe**: Catch errors early
- 🔄 **Retry logic**: Handle failures
- 📊 **Schemas**: Guide LLM output

### When to Use:
- ✅ Extracting structured data
- ✅ API responses
- ✅ Database inserts
- ✅ Integration with other systems

### Next: `03_prompt_engineering/04_prompt_templates.ipynb`